# BIOMASS stack: first inspection

Goal: verify the available stack folders and metadata **before loading any SLC arrays**.

This starter assumes a local `config/local.toml` has been created. The supplied
BIOMASS loader expects a specific primary/reference annotation and product order;
check that ordering before running `bmStack(...)`. The study stack reportedly has
7 acquisition folders, with the middle acquisition as the master.

In [ ]:
from pathlib import Path
from biomass_tomosar.paths import biomass_stack_dir

stack_root = biomass_stack_dir()
print("Stack folder:", stack_root)
if not stack_root.is_dir():
    raise FileNotFoundError(f"Configure [paths].biomass_stack in config/local.toml: {stack_root}")

products = sorted(p for p in stack_root.glob("BIO*") if p.is_dir())
print(f"Found {len(products)} products:")
for i, product in enumerate(products):
    print(f"  [{i}] {product.name}")

In [ ]:
# Inventory a product without reading 3 GB of imagery into RAM.
if products:
    example = products[0]
    print("Example product:", example.name)
    for dirname in ("annotation_primary", "annotation_coregistered", "annotation", "measurement", "preview"):
        folder = example / dirname
        print(f"  {dirname:25} exists={folder.exists()}")
    for path in sorted(example.rglob("*.xml"))[:5]:
        print("XML:", path.relative_to(example))

## Next: initialize the BIOMASS loader

The loader's `bmStack(baseDirs)` reads the annotation XML and LUTs, and assumes
`baseDirs[0]` provides the primary annotation. **Verify which folder contains the
correct primary annotation** and place it first. Other product order should be
kept documented, especially for interferogram baselines.

Do not call `stack.importSlcs()` until you have inspected image dimensions and
memory requirements. It loads the entire stack into RAM.

In [ ]:
# Uncomment only after validating the primary product and ordered list:
# from biomass_tomosar.bm_data_loader import bmStack
# ordered_products = [...]  # Path objects, with the correct primary product first
# stack = bmStack([str(p) for p in ordered_products])
# print("Tracks:", stack.Ntracks, "azimuth lines:", stack.nli, "range samples:", stack.nsa)